# 02 – EDA & Feature Engineering

**Fase CRISP-DM:** Data Preparation

## Objetivos desta fase
- Validar o risco de truncamento da janela de observação do LTV por cohort temporal
- Consolidar a EDA com foco em temporalidade, cardinalidade e multicolinearidade
- Construir o pipeline reproduzível de engenharia de recursos com `scikit-learn`
- Garantir que o `fit` do pipeline aconteça apenas no conjunto de treino


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    clean_raw_data,
    load_config,
    load_raw_data,
    split_data,
    validate_dtypes,
    validate_values,
)
from src.features import prepare_datasets_for_modeling

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(str(CONFIG_PATH))
RAW_PATH = PROJECT_ROOT / config["paths"]["raw_data"]


## Carga e validação inicial

Nesta etapa a base já entra limpa o suficiente para análise: `data_compra` em datetime, monetárias em `Float64`, categóricas em `string` e inteiras em `Int64`.


In [ ]:
df_raw = load_raw_data(RAW_PATH)
df, cleaning_report = clean_raw_data(df_raw)
validate_dtypes(df)
validate_values(df)

display(cleaning_report)
display(df.head())
display(df.dtypes)


## Visão geral e distribuição do alvo

In [ ]:
print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
display(df.describe(include="number").T)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["LTV"], bins=60, kde=True, ax=ax[0])
ax[0].set_title("LTV (escala linear)")
sns.histplot(np.log1p(df["LTV"]), bins=60, kde=True, ax=ax[1])
ax[1].set_title("log(1 + LTV)")
plt.tight_layout()


## Validação do tempo de observação

A base cobre clientes de **01/01/2023 a 31/05/2024**. O objetivo aqui é verificar se cohorts mais recentes têm menos tempo para acumular LTV, o que pode enviesar comparações diretas.


In [ ]:
periodo_min = df["data_compra"].min()
periodo_max = df["data_compra"].max()
print("Início da base:", periodo_min)
print("Fim da base:", periodo_max)
print("Dias totais de observação:", (periodo_max - periodo_min).days)


## Cohort temporal e truncamento da janela de LTV

A janela observável de cada cliente é calculada como `fim_da_base - data_compra`. Se cohorts recentes tiverem janelas muito menores, o LTV deles tende a estar truncado para baixo.


In [ ]:
df_cohort = df.copy()
df_cohort["cohort_mes"] = df_cohort["data_compra"].dt.to_period("M").astype(str)
df_cohort["janela_observacao_dias"] = (periodo_max - df_cohort["data_compra"]).dt.days.astype(int)
df_cohort["janela_menor_90d"] = df_cohort["janela_observacao_dias"] < 90

cohort_summary = (
    df_cohort.groupby("cohort_mes")
    .agg(
        clientes=("ID", "size"),
        ltv_medio=("LTV", "mean"),
        ltv_mediano=("LTV", "median"),
        janela_media_dias=("janela_observacao_dias", "mean"),
        janela_min_dias=("janela_observacao_dias", "min"),
        pct_janela_menor_90d=("janela_menor_90d", "mean"),
    )
    .reset_index()
)
cohort_summary["pct_janela_menor_90d"] = cohort_summary["pct_janela_menor_90d"] * 100

display(cohort_summary)


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
sns.lineplot(data=cohort_summary, x="cohort_mes", y="janela_media_dias", marker="o", ax=ax[0])
ax[0].set_title("Janela média de observação por cohort")
ax[0].tick_params(axis="x", rotation=45)

sns.lineplot(data=cohort_summary, x="cohort_mes", y="ltv_medio", marker="o", ax=ax[1], label="LTV médio")
sns.lineplot(data=cohort_summary, x="cohort_mes", y="ltv_mediano", marker="o", ax=ax[1], label="LTV mediano")
ax[1].set_title("LTV por cohort")
ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout()


### Leitura esperada
- Cohorts mais próximos de maio/2024 tendem a ter janelas médias menores.
- Se o LTV cair junto com a janela observada, isso é um indício forte de truncamento de observação.
- Na modelagem, isso reforça a necessidade de validação temporal e comparação cuidadosa entre clientes antigos e recentes.


## Alta cardinalidade: verificação após limpeza

As categorias raras já foram agrupadas durante a limpeza. Aqui confirmamos a cardinalidade remanescente antes do pipeline de encoding.


In [ ]:
for col in ["Produto Fonte", "Fonte Campanha", "Sexo", "Formacao"]:
    vc = df[col].value_counts(dropna=False)
    print(f"\n{col} | categorias: {vc.shape[0]}")
    display(vc.head(15).to_frame("qtd"))


## Multicolinearidade

Antes de modelar, avaliamos as numéricas. A correlação alta entre `valor_1_compra` e `recorrente_1_compra` já motivou a remoção de `recorrente_1_compra` da base de modelagem.


In [ ]:
corr_pred = df[["valor_1_compra", "recorrente_1_compra", "Renda"]].corr(numeric_only=True)
display(corr_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(corr_pred, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlação entre preditores numéricos")
plt.tight_layout()


## Split temporal

Como estamos em um problema sensível ao tempo, o split segue a ordem temporal. Isso evita vazamento de informação do futuro para o treino.


In [ ]:
train_df, val_df, test_df = split_data(
    df,
    test_size=config["data_split"]["test_size"],
    validation_size=config["data_split"]["validation_size"],
    random_state=config["data_split"]["random_state"],
    date_column="data_compra",
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "linhas": [len(train_df), len(val_df), len(test_df)],
        "min_data": [train_df["data_compra"].min(), val_df["data_compra"].min(), test_df["data_compra"].min()],
        "max_data": [train_df["data_compra"].max(), val_df["data_compra"].max(), test_df["data_compra"].max()],
    }
)
display(split_summary)


## Pipeline de engenharia de recursos

Decisões implementadas:
- `Renda` fica fora da modelagem.
- `recorrente_1_compra` sai por multicolinearidade com `valor_1_compra`.
- Categóricas usam `OneHotEncoder` com `handle_unknown='ignore'`, `drop='first'` e `min_frequency` para proteção extra contra cardinalidade.
- Numéricas usam `SimpleImputer` + `RobustScaler` ou `StandardScaler`.
- O alvo pode ser transformado com `log(LTV + 1)`.
- O `fit` do pipeline acontece **somente no treino**.


In [ ]:
prepared = prepare_datasets_for_modeling(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    scaling=config["features"]["scaling"],
    log_target=config["features"]["log_target"],
    categorical_min_frequency=config["features"]["categorical_min_frequency"],
)

display(prepared.metadata)
print("X_train:", prepared.X_train.shape)
print("X_val:", prepared.X_val.shape)
print("X_test:", prepared.X_test.shape)
display(prepared.X_train.head())
display(prepared.y_train.head())


In [ ]:
print("Renda nas features finais?", any(col == "Renda" for col in prepared.X_train.columns))
print("recorrente_1_compra nas features finais?", any(col == "recorrente_1_compra" for col in prepared.X_train.columns))
print("Mesmo número de colunas entre treino/val/test?", prepared.X_train.shape[1] == prepared.X_val.shape[1] == prepared.X_test.shape[1])


## Conclusões da fase
- Há evidência de risco de truncamento temporal do LTV para cohorts recentes, pois a janela de observação encurta no fim da base.
- As precauções contra alta cardinalidade foram tomadas na limpeza e reforçadas no `OneHotEncoder`.
- A multicolinearidade mais forte foi tratada removendo `recorrente_1_compra` da base de modelagem.
- `Renda` foi retirada da modelagem por baixa confiabilidade.
- O pipeline de features já está pronto e ajusta apenas no treino.

**Próxima fase:** modelagem com validação cruzada (`KFold=5`) e comparação de baseline vs. modelos avançados.
